## Imports

In [1]:
import requests
from dotenv import load_dotenv
import pandas as pd
import os
import numpy as np

## Phase 1: API Extraction

In [2]:
load_dotenv()
API_KEY= os.getenv("TMDB_API_KEY")

In [5]:
movie_ids = [299534, 19995, 140607, 299536, 597, 135397, 424, 259316, 24428, 1726, 283995, 118340, 168259, 76341, 209112, 271110, 120, 680, 559]

base_url = "https://api.themoviedb.org/3/movie/"
movies_data= []

for movie_id in movie_ids:

    url= f"{base_url}{movie_id}?api_key={API_KEY}"
    response= requests.get(url)

    if response.status_code == 200:
        movies_data.append(response.json())
    else:
        print(f"Failed to Fetch ID {movie_id}: {response.status_code}")


df= pd.DataFrame(movies_data)

df.head(2)

,adult,backdrop_path,belongs_to_collection,budget,genres,homepage,id,imdb_id,origin_country,original_language,...,revenue,runtime,softcore,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,/7RyHsO4yDXtBv1zUU3mTpHeQ0d5.jpg,"{'id': 86311, 'name': 'The Avengers Collection...",356000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 878, ...",https://www.marvel.com/movies/avengers-endgame,299534,tt4154796,[US],en,...,2927449455,181,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Avenge the fallen.,Avengers: Endgame,False,8.246,29005
1,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,"{'id': 87096, 'name': 'Avatar Collection', 'po...",237000000,"[{'id': 878, 'name': 'Science Fiction'}, {'id'...",https://www.avatar.com/movies/avatar,19995,tt0499549,[US],en,...,2923706026,162,False,"[{'english_name': 'English', 'iso_639_1': 'en'...",Released,Enter the world of Pandora.,Avatar,False,7.611,34845


## Phase 2 Data Cleaning

In [6]:
import ast

def extract_names(json_data):



    try:

        if isinstance(json_data, list):
            return "|".join([item['name'] for item in json_data])

        elif isinstance(json_data,str):
            json_data= ast.literal_eval(json_data)

        else:
            return ""

        names= [item['name'] for item in json_data]

        return "|".join(names)
    
    except:
        return ""


df['genres']= df['genres'].apply(extract_names)
df['production_companies']= df['production_companies'].apply(extract_names)

df[['title', 'genres', 'production_companies']].head()
    

,title,genres,production_companies
0,Avengers: Endgame,Adventure|Science Fiction|Action,Marvel Studios
1,Avatar,Science Fiction|Action|Adventure,Dune Entertainment|Lightstorm Entertainment|20...
2,Star Wars: The Force Awakens,Adventure|Action|Science Fiction,Lucasfilm Ltd.|Bad Robot
3,Avengers: Infinity War,Adventure|Action|Science Fiction,Marvel Studios
4,Titanic,Drama|Romance,Paramount Pictures|20th Century Fox|Lightstorm...


In [7]:
df['budget']= pd.to_numeric(df['budget'], errors='coerce')
df['revenue']= pd.to_numeric(df['revenue'], errors='coerce')

df['budget_musd'] = df['budget'] / 1000000
df['revenue_musd'] = df['revenue'] / 1000000

df[['title', 'budget_musd', 'revenue_musd']].head()

,title,budget_musd,revenue_musd
0,Avengers: Endgame,356.0,2927.449455
1,Avatar,237.0,2923.706026
2,Star Wars: The Force Awakens,245.0,2068.223624
3,Avengers: Infinity War,300.0,2052.415039
4,Titanic,200.0,2264.162353


In [8]:
columns_to_drop = ['adult', 'imdb_id', 'original_title', 'video', 'homepage']
df.drop(columns=columns_to_drop, errors="ignore", inplace= True)
df= df[df['status']== 'Released'].copy()
df.drop(columns=['status'], inplace= True)

df['release_date']= pd.to_datetime(df['release_date'], errors="coerce")
df[['title','release_date']].head()

,title,release_date
0,Avengers: Endgame,2019-04-24
1,Avatar,2009-12-16
2,Star Wars: The Force Awakens,2015-12-15
3,Avengers: Infinity War,2018-04-25
4,Titanic,1997-12-18


In [9]:
df[['budget', 'revenue', 'runtime']]= df[['budget', 'revenue', 'runtime']].replace(0, np.nan)
df['spoken_languages'] = df['spoken_languages'].apply(extract_names)
df['production_countries'] = df['production_countries'].apply(extract_names)


def get_collection_name(json_data):
    try:
        if isinstance(json_data, dict):
            return json_data.get('name','')
        elif isinstance(json_data,str):
            parsed= ast.literal_eval(json_data)
            return parsed.get('name','')
        else:
            return ""
    except:
        return ""

df['belongs_to_collection']= df['belongs_to_collection'].apply(get_collection_name)
df['belongs_to_collection'].head()

0    The Avengers Collection
1          Avatar Collection
2       Star Wars Collection
3    The Avengers Collection
4                           
Name: belongs_to_collection, dtype: str

In [11]:
df.dropna(thresh=10, inplace=True)

df.drop_duplicates(subset=['id'] ,inplace=True)

final_order = [
    'id', 'title', 'tagline', 'release_date', 'genres', 'belongs_to_collection',
    'original_language', 'budget_musd', 'revenue_musd', 'production_companies',
    'production_countries', 'vote_count', 'vote_average', 'popularity', 'runtime',
    'overview', 'spoken_languages', 'poster_path'
]

existing_columns= [col for col in final_order if col in df.columns]
df= df[existing_columns]

df.reset_index( drop = True,inplace= True)

df.head(2)

,id,title,tagline,release_date,genres,belongs_to_collection,original_language,budget_musd,revenue_musd,production_companies,production_countries,vote_count,vote_average,popularity,runtime,overview,spoken_languages,poster_path
0,299534,Avengers: Endgame,Avenge the fallen.,2019-04-24,Adventure|Science Fiction|Action,The Avengers Collection,en,356.0,2927.449455,Marvel Studios,United States of America,29005,8.246,87.1502,181,After the devastating events of Avengers: Infi...,English|日本語|,/bR8ISy1O9XQxqiy0fQFw2BX72RQ.jpg
1,19995,Avatar,Enter the world of Pandora.,2009-12-16,Science Fiction|Action|Adventure,Avatar Collection,en,237.0,2923.706026,Dune Entertainment|Lightstorm Entertainment|20...,United States of America|United Kingdom,34845,7.611,48.3380,162,"In the 22nd century, a paraplegic Marine is di...",English|Español,/gKY6q7SjCkAU6FqvqWybDYgUKIF.jpg
